# M5 HPO · Chronos-2 Fine-Tuned Smoothness-Ensemble
Tunes one context length per demand segment **plus** shared fine-tuning
hyperparameters (steps, mode, lr, batch size) and covariate type.

Each Optuna trial runs **4 separate Chronos-2 models** — one per segment — on the
full training set, filters each forecast to its segment's IDs, merges all four
into one ensemble DF and evaluates the combined WRMSSE.

Setting `ft_steps = 0` in `FT_STEPS_CHOICES` makes those trials equivalent to
zero-shot runs and uses a shortened tag so they share cache with ZS experiments.

Per-`(segment, CL, ft_steps, mode, lr, bs, cov)` forecast caching means any
repeated combination across trials is loaded from disk, not re-inferred.

# 1 · Imports

In [1]:
import os
import sys
sys.path.append("/home/nmwamsojo/tsfm-explo/src/jobs/")

import gc
import json
import torch
import optuna
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from IPython.display import display

from m5_dataprep import M5DataPipeline
from m5_exploration import (
    M5ExplorationSuite,
    DEFAULT_CHRONOS_CONFIG,
)
from m5_evaluator import M5Evaluator

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

/home/nmwamsojo/tsfm-explo/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CUDA available: True
  GPU 0: Quadro RTX 5000
  GPU 1: Quadro RTX 5000


In [2]:
os.environ["OMP_NUM_THREADS"]        = "1"
os.environ["MKL_NUM_THREADS"]        = "1"
os.environ["OPENBLAS_NUM_THREADS"]   = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"]    = "1"
os.environ["CUDA_VISIBLE_DEVICES"]   = "0,1"

INFERENCE_DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"

# 2 · Config
**Edit this cell only** to change experiment settings.

In [3]:
# ── Data ──────────────────────────────────────────────────────────────────────
DATA_PATH     = "/mnt/lab/datasets/M5/jointed_M5.parquet"
CALENDAR_PATH = "/mnt/lab/nmwamsojo/m5_data/calendar.csv"
ACTUALS_PATH  = "/mnt/lab/nmwamsojo/m5_data/sales_test_evaluation.csv"

# Flip this to switch between prepared datasets:
#   "sales_only"  — standard M5 setup, no extra feature engineering
#   "default"     — includes all engineered columns (is_friday, …)
DATA_TAG = "sales_only"

CUTOFF_DAY = (
    pd.to_datetime("2016-05-22") - pd.Timedelta(days=28)
).strftime("%Y-%m-%d")
print(f"Cutoff day : {CUTOFF_DAY}")
print(f"Data tag   : {DATA_TAG}")

# ── Context-length search space (per segment) ─────────────────────────────────
CL_CHOICES = [2**i for i in range(10)]   # [1, 2, 4, 8, ..., 1024]

# ── Fine-tuning search space (shared across all segments per trial) ───────────
FT_STEPS_CHOICES = [0, 100, 200, 500, 1000, 1500, 2000, 3000]  # 0 = zero-shot (no gradient updates)
FT_MODE_CHOICES  = ["lora", "full"]           # ignored when ft_steps == 0
FT_LR_CHOICES    = [1e-5, 5e-5, 1e-4, 5e-4, 1e-3, 5e-3, 1e-2]
FT_BATCH_CHOICES = [32, 64, 128, 256]         # fine-tune training batch size

BATCH_SIZE = 64    # inference batch size — fixed
N_TRIALS   = 100    # Optuna budget (FT trials are expensive; start with 50)

# ── Covariate search space (shared across all segments per trial) ─────────────
# DATA_TAG cache already contains all columns; no re-preparation needed.
COVARIATE_OPTIONS = {
    #"only":        [],
    "is_weekend":  ["is_friday", "is_saturday", "is_sunday"],
    "event":       ["event_name_1", "event_type_1"],
    "price":       ["sell_price"],
    'snap' :       ['snap_CA', 'snap_TX', 'snap_WI'],
    "is_weekend_event":  ["is_friday", "is_saturday", "is_sunday", "event_name_1", "event_type_1"],
    "is_weekend_price":  ["is_friday", "is_saturday", "is_sunday", "sell_price"],
    "event_price": ["event_name_1", "event_type_1", "sell_price"],

    # SNAP + Single Core Features
    "is_weekend_snap": ["is_friday", "is_saturday", "is_sunday", "snap_CA", "snap_TX", "snap_WI"],
    "event_snap":      ["event_name_1", "event_type_1", "snap_CA", "snap_TX", "snap_WI"],
    "price_snap":      ["sell_price", "snap_CA", "snap_TX", "snap_WI"],

    # SNAP + Multi-Core Combinations
    "is_weekend_event_snap": ["is_friday", "is_saturday", "is_sunday", "event_name_1", "event_type_1", "snap_CA", "snap_TX", "snap_WI"],
    "is_weekend_price_snap": ["is_friday", "is_saturday", "is_sunday", "sell_price", "snap_CA", "snap_TX", "snap_WI"],
    "event_price_snap":      ["event_name_1", "event_type_1", "sell_price", "snap_CA", "snap_TX", "snap_WI"],

    "all": ["is_friday", "is_saturday", "is_sunday", "event_name_1", "event_type_1", "sell_price", "snap_CA", "snap_TX", "snap_WI"],
}
COV_CHOICES = list(COVARIATE_OPTIONS.keys())

n_seg = 4   # updated after segment cell runs; placeholder for the print below
print(f"CL choices       : {CL_CHOICES}")
print(f"FT steps         : {FT_STEPS_CHOICES}")
print(f"FT modes         : {FT_MODE_CHOICES}")
print(f"FT lr            : {FT_LR_CHOICES}")
print(f"FT batch         : {FT_BATCH_CHOICES}")
print(f"Cov options      : {COV_CHOICES}")
print(f"Inference batch  : {BATCH_SIZE}  (fixed)")
print(f"Trials budget    : {N_TRIALS}")

# ── AutoGluon wrapper ─────────────────────────────────────────────────────────
WRAPPER = {
    "eval_metric":          "RMSSE",
    "enable_ensemble":      False,
    "skip_model_selection": True,
    "verbosity":            1,
}

# ── Base Chronos config — all tunable fields overridden per trial ─────────────
CFG_CHRONOS_BASE = {
    **DEFAULT_CHRONOS_CONFIG,
    "use_static": False,
    "device":     INFERENCE_DEVICE,
}

Cutoff day : 2016-04-24
Data tag   : sales_only
CL choices       : [1, 2, 4, 8, 16, 32, 64, 128, 256, 512]
FT steps         : [0, 100, 200, 500, 1000, 1500, 2000, 3000]
FT modes         : ['lora', 'full']
FT lr            : [1e-05, 5e-05, 0.0001, 0.0005, 0.001, 0.005, 0.01]
FT batch         : [32, 64, 128, 256]
Cov options      : ['is_weekend', 'event', 'price', 'snap', 'is_weekend_event', 'is_weekend_price', 'event_price', 'is_weekend_snap', 'event_snap', 'price_snap', 'is_weekend_event_snap', 'is_weekend_price_snap', 'event_price_snap', 'all']
Inference batch  : 64  (fixed)
Trials budget    : 100


# 3 · Data

In [4]:
pipeline = M5DataPipeline(config={"tag": DATA_TAG})
hist_df, hist_df_trimmed, future_df, static_df, weights_scales = (
    pipeline.get_prepared_data(DATA_PATH, CUTOFF_DAY, level=12, force_reprepare=False)
)

print(f"\nhist_df         : {hist_df.shape}  cols: {list(hist_df.columns)}")
print(f"hist_df_trimmed : {hist_df_trimmed.shape}")
print(f"future_df       : {future_df.shape}")
print(f"static_df       : {static_df.shape}")
print(f"weights_scales  : {weights_scales.shape}  levels: {sorted(weights_scales['level'].unique())}")

del pipeline
gc.collect()

--- Cache Hit: Data found in /mnt/lab/nmwamsojo/prepared_data/sales_only/level_12/20160424 ---



hist_df         : (58327370, 21)  cols: ['id', 'date', 'sales_quantity', 'wm_yr_wk', 'wday', 'month', 'year', 'event_name_1', 'event_type_1', 'snap_CA', 'snap_TX', 'snap_WI', 'sell_price', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id', 'is_friday', 'is_saturday', 'is_sunday']
hist_df_trimmed : (45942500, 21)
future_df       : (853720, 20)
static_df       : (30490, 6)
weights_scales  : (42840, 7)  levels: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12)]


20

In [5]:
def _wide_to_long(gt_wide: pd.DataFrame, calendar_path: str) -> pd.DataFrame:
    if "id" not in gt_wide.columns:
        gt_wide["id"] = gt_wide["item_id"] + "_" + gt_wide["store_id"] + "_evaluation"
    day_cols = [c for c in gt_wide.columns if c.startswith("d_")]
    long = gt_wide.melt(id_vars=["id"], value_vars=day_cols,
                        var_name="d", value_name="sales_quantity")
    cal = pd.read_csv(calendar_path, usecols=["d", "date"])
    cal["date"] = pd.to_datetime(cal["date"])
    long = long.merge(cal, on="d", how="left").drop(columns=["d"])
    long["id"] = long["id"].str.replace("_evaluation", "", regex=False)
    return long[["id", "date", "sales_quantity"]]


if CUTOFF_DAY == "2016-05-22":
    _eval_raw = pd.read_csv(ACTUALS_PATH)
    df_actual = _wide_to_long(_eval_raw, CALENDAR_PATH)
    del _eval_raw
else:
    df_actual = (
        pd.read_parquet(DATA_PATH, columns=["id", "date", "sold"])
        .rename(columns={"sold": "sales_quantity"})
    )
    df_actual["id"] = (
        df_actual["id"].astype(str)
        .str.replace("_evaluation", "", regex=False)
        .str.replace("_validation",  "", regex=False)
    )

print(f"Actuals : {df_actual.shape}  |  "
      f"{df_actual['date'].min().date()} \u2192 {df_actual['date'].max().date()}")

Actuals : (59181090, 3)  |  2011-01-29 → 2016-05-22


# 4 · Evaluator & Suite
Initialised **once** — reused across all trials.

In [6]:
evaluator = M5Evaluator(
    raw_train_df     = hist_df,
    trimmed_train_df = hist_df_trimmed,
    static_df        = static_df,
    weights_df       = weights_scales,
    target_col       = "sales_quantity",
    price_col        = "sell_price",
)
suite = M5ExplorationSuite(
    horizon  = 28,
    ag_path  = "/mnt/lab/nmwamsojo/autogluon_models/explorations",
    base_dir = "/mnt/lab/nmwamsojo/prepared_data",
)

  [CPU] CuPy not available — scale computation on 18 CPU cores.
  Building hierarchy scales and weights …


# 5 · Segment ID Map
Switch `seg_columns` between `"smoothness_segs"` and `"weight_segs"` to change
the segmentation scheme used for the ensemble.

In [7]:
quantile_labels = ["Low", "Medium-Low", "Medium-High", "High"]

weights_scales["weight_segs"] = pd.qcut(
    weights_scales["weight"],
    q=4,
    labels=quantile_labels,
)

In [ ]:
seg_columns  = "weight_segs"   # swap to "smoothness_segs" to use demand-pattern segments
seg_names    = ["Smooth", "Erratic", "Intermittent", "Lumpy"]
undefined_in = "Lumpy"

if "weight" in seg_columns:
    seg_names    = quantile_labels
    undefined_in = "Low"

# Scheme identifier used in cache tags — shared across all classes with the same
# (CL, ft_params, cov_type) so a single forecast file is reused per hyperparameter combo.
seg_scheme = seg_columns.replace("_segs", "")   # "weight" or "smoothness"

if seg_columns in hist_df_trimmed.columns:
    _seg_src = hist_df_trimmed[["id", seg_columns]].drop_duplicates()
else:
    _seg_src = (
        weights_scales[weights_scales["level"] == 12][["id", seg_columns]]
        .dropna(subset=[seg_columns])
    )

seg_id_map = {
    seg: _seg_src[_seg_src[seg_columns] == seg]["id"].tolist()
    for seg in seg_names
}
_undefined = _seg_src[_seg_src[seg_columns] == "Undefined"]["id"].tolist()
seg_id_map[undefined_in].extend(_undefined)

del _seg_src, _undefined

total = sum(len(v) for v in seg_id_map.values())
for seg, ids in seg_id_map.items():
    print(f"  {seg:>14}: {len(ids):>6,} series  ({100*len(ids)/total:.1f}%)")
print(f"  {'TOTAL':>14}: {total:>6,}")
print(f"\n  seg_scheme = '{seg_scheme}'")

# 6 · Optuna HPO
Each trial samples:
- One `context_length` per segment (4 params)
- Shared `ft_steps`, `ft_mode`, `ft_lr`, `ft_bs`, `cov_type` (5 params)

When `ft_steps == 0` the trial is equivalent to zero-shot; `ft_mode`/`ft_lr`/`ft_bs`
are still sampled by Optuna but excluded from the cache tag so all zero-shot runs
with the same `(segment, CL, cov_type)` share one forecast file.

> **Tip**: for long studies, persist the Optuna study to a SQLite DB so you can
> resume after kernel restarts:
> `optuna.create_study(storage="sqlite:////mnt/lab/nmwamsojo/optuna_ft.db", load_if_exists=True, ...)`

In [ ]:
# Per-segment CL params + shared FT/cov params logged in results
SEG_PARAM     = {seg: f"CL_{seg.lower().replace('-', '_')}" for seg in seg_names}
SHARED_PARAMS = ["ft_steps", "ft_mode", "ft_lr", "ft_bs", "cov_type"]


def _make_seg_tag(seg_scheme: str, cl: int,
                  ft_steps: int, ft_mode: str, ft_lr: float,
                  ft_bs: int, cov_type: str) -> str:
    """Deterministic cache tag shared across all segment classes with the same hyperparams.

    ZS (ft_steps=0) tags match hpo_zs_withcov_ensembles.ipynb → shared cache between studies.
    """
    if ft_steps == 0:
        return f"hpo_zs_{seg_scheme}_cl{cl}_{cov_type}"
    lr_str = f"{ft_lr:.0e}".replace("-0", "-")
    return (
        f"hpo_ft_{seg_scheme}_cl{cl}"
        f"_steps{ft_steps}_{ft_mode}_lr{lr_str}_bs{ft_bs}_{cov_type}"
    )


def objective(trial: optuna.Trial) -> float:
    # ── 1. Sample hyperparameters ─────────────────────────────────────────────
    seg_cl = {
        seg: trial.suggest_categorical(param, CL_CHOICES)
        for seg, param in SEG_PARAM.items()
    }
    ft_steps = trial.suggest_categorical("ft_steps", FT_STEPS_CHOICES)
    ft_mode  = trial.suggest_categorical("ft_mode",  FT_MODE_CHOICES)
    ft_lr    = trial.suggest_categorical("ft_lr",    FT_LR_CHOICES)
    ft_bs    = trial.suggest_categorical("ft_bs",    FT_BATCH_CHOICES)
    cov_type = trial.suggest_categorical("cov_type", COV_CHOICES)
    known_cov_cols = COVARIATE_OPTIONS[cov_type]

    print(
        f"\n[Trial {trial.number}] "
        f"CLs={{{', '.join(f'{p}={seg_cl[s]}' for s, p in SEG_PARAM.items())}}}  "
        f"steps={ft_steps} mode={ft_mode} lr={ft_lr:.0e} bs={ft_bs} cov={cov_type}"
    )

    seg_forecasts = []
    try:
        # ── 2. Run one Chronos model per segment ─────────────────────────────
        for seg_name, cl in seg_cl.items():
            seg_tag = _make_seg_tag(seg_scheme, cl, ft_steps, ft_mode, ft_lr, ft_bs, cov_type)
            ids     = set(seg_id_map[seg_name])

            fcst = suite.run(
                hist_df      = hist_df_trimmed,
                future_df    = future_df,
                static_df    = static_df,
                model        = "Chronos2",
                exp_config   = {
                    **CFG_CHRONOS_BASE,
                    "context_length":       cl,
                    "fine_tune_steps":      ft_steps,
                    "fine_tune_mode":       ft_mode,
                    "fine_tune_lr":         ft_lr,
                    "fine_tune_batch_size": ft_bs,
                    "batch_size":           BATCH_SIZE,
                    "known_cov_cols":       known_cov_cols,
                },
                exp_tag      = seg_tag,
                data_tag     = DATA_TAG,
                cutoff_day   = CUTOFF_DAY,
                wrapper_dict = WRAPPER,
                force_run    = False,
            )

            seg_forecasts.append(fcst[fcst["id"].isin(ids)].copy())
            del fcst

        # ── 3. Merge all segment forecasts → one ensemble DF ─────────────────
        ensemble_fcst = pd.concat(seg_forecasts, ignore_index=True)
        seg_forecasts.clear()

        # ── 4. Evaluate ───────────────────────────────────────────────────────
        metrics = evaluator.evaluate_all(ensemble_fcst, df_actual)
        del ensemble_fcst

        for seg, param in SEG_PARAM.items():
            trial.set_user_attr(param, seg_cl[seg])
        trial.set_user_attr("ft_steps", ft_steps)
        trial.set_user_attr("ft_mode",  ft_mode)
        trial.set_user_attr("ft_lr",    ft_lr)
        trial.set_user_attr("ft_bs",    ft_bs)
        trial.set_user_attr("cov_type", cov_type)
        trial.set_user_attr("WRMSSE",   metrics["WRMSSE"])

        print(f"[Trial {trial.number}] WRMSSE = {metrics['WRMSSE']:.4f}")

        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        return metrics["WRMSSE"]

    except Exception as e:
        print(f"[Trial {trial.number}] failed: {e}")
        seg_forecasts.clear()
        gc.collect()
        return float("inf")

In [ ]:
optuna.logging.set_verbosity(optuna.logging.WARNING)

# To persist across kernel restarts, pass a storage URL, e.g.:
# storage = "sqlite:////mnt/lab/nmwamsojo/optuna_ft_ensemble.db"
study = optuna.create_study(
    direction  = "minimize",
    sampler    = optuna.samplers.TPESampler(seed=42),
    study_name = "chronos_ft_ensemble",
    # storage  = storage,
    # load_if_exists = True,
)
study.optimize(objective, n_trials=N_TRIALS, n_jobs=1, gc_after_trial=True)

print(f"\nBest WRMSSE : {study.best_value:.4f}")
print(f"Best params : {study.best_params}")


[Trial 0] CLs={CL_low=2, CL_medium_low=2, CL_medium_high=32, CL_high=16}  steps=500 mode=lora lr=1e-05 bs=256 cov=event_snap
[hpo_ft_ens_fitAll_low_cl2_steps500_lora_lr1e-5_bs256_event_snap] Forecasts not found at /mnt/lab/nmwamsojo/prepared_data/sales_only/level_12/20160424/models/hpo_ft_ens_fitAll_low_cl2_steps500_lora_lr1e-5_bs256_event_snap/forecasts.parquet…
[hpo_ft_ens_fitAll_low_cl2_steps500_lora_lr1e-5_bs256_event_snap] Running Chronos2 experiment…
  known covariates : ['event_name_1', 'event_type_1', 'snap_CA', 'snap_TX', 'snap_WI']
fit known_cov_cols : ['event_name_1', 'event_type_1', 'snap_CA', 'snap_TX', 'snap_WI']
Fitting model with hyperparameters: 

	Chronos2: {'model_path': 'autogluon/chronos-2', 'context_length': 2, 'device': 'cuda:0', 'batch_size': 64, 'fine_tune_steps': 500, 'fine_tune_mode': 'lora', 'fine_tune_lr': 1e-05, 'fine_tune_batch_size': 256}


# 7 · Results

In [ ]:
df_trials = study.trials_dataframe(attrs=("number", "value", "params", "user_attrs"))
df_trials = df_trials.sort_values("value").reset_index(drop=True)
df_trials.rename(columns={"value": "WRMSSE"}, inplace=True)

cl_cols     = [f"params_{p}" for p in SEG_PARAM.values()]
shared_cols = [f"params_{p}" for p in SHARED_PARAMS]
param_cols  = cl_cols + shared_cols

display(
    df_trials[["number", "WRMSSE"] + param_cols]
    .head(20)
    .style
    .format({"WRMSSE": "{:.4f}"})
    .background_gradient(subset=["WRMSSE"], cmap="RdYlGn_r")
    .set_caption("Top-20 trials by ensemble WRMSSE (lower is better)")
)

print(f"\nBest  WRMSSE = {df_trials['WRMSSE'].min():.4f}  "
      f"(trial #{int(df_trials.iloc[0]['number'])})")
print(f"Worst WRMSSE = {df_trials['WRMSSE'].max():.4f}")

# ── Summaries by shared params ────────────────────────────────────────────────
for col in ["params_ft_steps", "params_ft_mode", "params_cov_type"]:
    if col in df_trials.columns:
        print(f"\nMean WRMSSE by {col.replace('params_', '')}:")
        print(
            df_trials.groupby(col)["WRMSSE"]
            .agg(["mean", "min", "count"])
            .sort_values("mean")
            .to_string()
        )

In [ ]:
# Per-segment CL vs WRMSSE
n_segs = len(SEG_PARAM)
ncols  = min(n_segs, 2)
nrows  = (n_segs + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(11, 4 * nrows), sharey=True)
axes = axes.flatten() if n_segs > 1 else [axes]

for ax, (seg, param) in zip(axes, SEG_PARAM.items()):
    col     = f"params_{param}"
    grouped = df_trials.groupby(col)["WRMSSE"].agg(["mean", "min"]).reset_index()
    ax.plot(grouped[col], grouped["mean"], marker="o", label="mean")
    ax.plot(grouped[col], grouped["min"],  marker="s", linestyle="--", label="min")
    ax.set_xscale("log", base=2)
    ax.set_title(seg, fontweight="bold")
    ax.set_xlabel("Context Length")
    ax.set_ylabel("Ensemble WRMSSE")
    ax.xaxis.set_major_formatter(mticker.ScalarFormatter())
    ax.legend(frameon=False, fontsize=8)
    ax.grid(True, alpha=0.3)

for ax in axes[n_segs:]:
    ax.set_visible(False)

fig.suptitle("Ensemble WRMSSE vs Context Length per Segment",
             fontsize=13, fontweight="bold", y=1.01)
fig.tight_layout()
plt.show()

In [ ]:
from pathlib import Path

out = Path(f"/mnt/lab/nmwamsojo/results/m5_hpo_ft_ensemble_{CUTOFF_DAY.replace('-', '')}.csv")
out.parent.mkdir(parents=True, exist_ok=True)
df_trials.to_csv(out, index=False)
print(f"Results saved \u2192 {out}  ({len(df_trials)} rows)")